## Part 3: Advanced Text Processing - LDA and BERTopic Topic Modeling (20 pts)

**Resources:**
- LDA:
    - https://medium.com/sayahfares19/text-analysis-topic-modelling-with-spacy-gensim-4cd92ef06e06 
    - https://www.kaggle.com/code/faressayah/text-analysis-topic-modeling-with-spacy-gensim#%F0%9F%93%9A-Topic-Modeling (code for previous post)
    - https://towardsdatascience.com/topic-modelling-in-python-with-spacy-and-gensim-dc8f7748bdbf/ 
- BERTopic:
    - https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_documents.html#visualize-documents-with-plotly 
    - https://maartengr.github.io/BERTopic/getting_started/visualization/visualize_topics.html 


In [37]:
from spacy import displacy
from bertopic import BERTopic
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from sklearn.feature_extraction.text import CountVectorizer
import pyLDAvis
import pyLDAvis.gensim_models
import spacy
import pandas as pd

from tqdm import tqdm
from collections import Counter


nlp = spacy.load("en_core_web_sm")


import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-dark')

# read in SOTU.csv
sou = pd.read_csv('data/SOTU.csv')


### LDA

- Train an LDA model with 18 topics
- Output the top 10 words for each topic. 
- Output the topic distribution for the first speech
- Make a visualization

You may use the next two cells to process the data.

In [38]:
def preprocess_text(text): 
    doc = nlp(text) 
    return [token.lemma_.lower() for token in doc if not token.is_stop and not token.is_punct and not token.is_space and len(token.lemma_) > 3]

In [ ]:
# Process all texts - note this takes ~ 5 minutes to run
processed_docs = sou['Text'].apply(preprocess_text)

To train an LDA model, use the LdaModel function that we imported a couple of cells back. The last resource linked under the LDA section is especially useful for walking through the steps we have below. *Note: one of the arguments to the LdaModel function is `random_state` which specifies the random seed for reproducibility. Please set yours to 42. Further, the last resource provided uses `LdaMulticore` which is essentially a parallelizable version of our function `LdaModel`. Use `LdaModel` instead, but the usage will be similar, except you can ignore the `iterations` and `workers` arguments..*.

In [ ]:
# Build dictionary from processed_docs, which is a list of tokens extracted from our speeches
dictionary = Dictionary(processed_docs)
# Here I convert to the bag of words corpus
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]


In [ ]:
# train LDA model with 18 topics
lda_model = LdaModel(corpus=corpus, id2word=dictionary, num_topics=18, random_state=42, passes=10)


In [ ]:
# print the top 10 words for each topic

print("--- LDA Topics ---")

# loop over each topic in the LDA
for topic_id, topic_str in lda_model.print_topics(num_topics=-1):
    # try and maintain the output format shown but I did notice that the example and my output are different
    print(f"Topic: {topic_id} \nWords: {topic_str}\n")


In [ ]:
# print the topic distribution for the first speech
#using float 32 for formating
from numpy import float32
#selects the one with the max probabiltiy
dominant_topic = max(lda_model.get_document_topics(corpus[0]), key=lambda x: x[1])
# I format the outputs as tuples with topic idea and trying to use float32 to get output forat
formatted_output = [(dominant_topic[0], float32(dominant_topic[1]))]
print(formatted_output)



In [ ]:
# make a visualization using pyLDAvis
pyLDAvis.enable_notebook()

lda_vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary)

lda_vis


### BERTopic

- Train a BERTopic model with a `min_topic_size` of 3 *Hint: use `BERTopic` to instantiate the model and specify `min_topic_size` in here. Actually fit the model using `fit_transform`, which `docs` passed into this.*
- Output the top 10 words for each topic. 
- Output the topic distribution for the first speech
- Make a visualization of the topics (see topic_model.visualize_topics())

In [ ]:
docs = sou['Text'].to_list()

In [ ]:
# train the model - this takes about 30 seconds

# remove stop words from the topics (Hint: use CountVectorizer and then .update_topics on topic_model)
docs = sou['Text'].to_list()
topic_model = BERTopic(min_topic_size=3)
# fit the bert model and get probs
topics, probs = topic_model.fit_transform(docs)

# I will remove stop words using CountVectorizer
vectorizer_model = CountVectorizer(stop_words="english")
#update topics so that the stop words are removed
topic_model.update_topics(docs, vectorizer_model=vectorizer_model)


In [ ]:
# output the top 10 words for each topic - hint see get_topic_info
topic_info = topic_model.get_topic_info()
# I loop over the topic id in the df
for topic_id in topic_info['Topic']:
# Using -1 to skip the noisy topics
    if topic_id == -1:
        continue 
    print(f"Topic {topic_id}: {topic_model.get_topic(topic_id)}\n")


In [ ]:
# output the topic distribution for the first speech
# hint: check out approximate_distribution() and visualize_distribution()
first_doc = docs[0]
# herre I get the topic probabolioties for the first document and I ignore the second value
dist, _ = topic_model.approximate_distribution(first_doc)
print(dist)
topic_model.visualize_distribution(dist[0])


In [ ]:
# run this cell to visualize the topics
topic_model.visualize_topics()